In [1]:
RUN_NAME = "localiser_run4_vgg16enc_patch"
ENCODER = "vgg16"          # "vgg16" -> localise.build_unet_pretrained; None -> scratch build_unet (run-3 arch)
WHOLE_IMAGE = False        # False = L4 (512-px lesion-oversampled patches, batch 8)
                           # True  = L4b attribution control (whole 1024x576, batch 4) - NOT launched by default
EPOCHS = 100

MERGED_STORE = "/kaggle/input/datasets/tataruteodor/ddsm-localiser-tensors-merged"   # <- adjust to the attached dataset path
LESION_STORE = "/kaggle/input/datasets/tataruteodor/localiser-tensors"          # <- adjust
SRC_DATASET  = "/kaggle/input/datasets/tataruteodor/ddsm-src"                        # <- adjust (contains src/)
OUT = f"/kaggle/working/{RUN_NAME}"

In [2]:
import os, shutil, sys, json
import tensorflow as tf

if os.path.exists("/kaggle/working/src"):
    shutil.rmtree("/kaggle/working/src")
shutil.copytree(os.path.join(SRC_DATASET, "src"), "/kaggle/working/src")
os.chdir("/kaggle/working")
sys.path.insert(0, "/kaggle/working")

print("TF", tf.__version__, "| GPUs", tf.config.list_physical_devices("GPU"))
tf.keras.mixed_precision.set_global_policy("float32")
assert tf.keras.mixed_precision.global_policy().name == "float32", "run 4 is float32 by design"

from src import config, localise
m = localise.build_unet_pretrained(ENCODER) if ENCODER else localise.build_unet()
print(f"{m.name}: {m.count_params():,} params")
del m
for d in (MERGED_STORE, LESION_STORE):
    print(d, sorted(os.listdir(d)))

TF 2.20.0 | GPUs [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


I0000 00:00:1788856933.346758      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
unet_vgg16_localiser: 31,410,497 params
/kaggle/input/datasets/tataruteodor/ddsm-localiser-tensors-merged ['train_images.npy', 'train_masks.npy', 'train_meta.csv', 'val_images.npy', 'val_masks.npy', 'val_meta.csv']
/kaggle/input/datasets/tataruteodor/localiser-tensors ['test_images.npy', 'test_masks.npy', 'test_meta.csv', 'train_images.npy', 'train_masks.npy', 'train_meta.csv', 'val_images.npy', 'val_masks.npy', 'val_meta.csv']


In [3]:
from src.train_localiser_l4 import main
manifest = main(MERGED_STORE, LESION_STORE, OUT,
                encoder=ENCODER or "none", whole_image=WHOLE_IMAGE,
                epochs=EPOCHS, run_name=RUN_NAME)
print(json.dumps({k: manifest[k] for k in (
    "run", "regime", "params", "steps_per_epoch", "epochs_run", "first_nan_epoch",
    "best_epoch", "val_hard_iou_at_best_epoch", "val_dice_at_best_epoch",
    "val_loss_at_best_epoch", "best_valloss_epoch", "max_val_hard_iou_any_epoch",
    "wall_time_s") if k in manifest}, indent=2))

[l4] train store (1146, 1024, 576) float16 (merged, one row per mammogram) | val store (243, 1024, 576) float16 (per lesion) | output prior 0.00497
[l4] unet_vgg16_localiser: 31,410,497 params (14,714,688 encoder at 0.0001, 16,685,825 decoder at 0.001) | regime patch512 | batch 8 | 287 steps/epoch x 100
Epoch 1/100


2026-09-08 08:44:57.848922: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv (f32[32,96,3,3]{3,2,1,0}, u8[0]{0}) custom-call(f32[8,96,512,512]{3,2,1,0}, f32[8,32,512,512]{3,2,1,0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardFilter", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]} is taking a while...
2026-09-08 08:44:59.088167: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 2.239390789s
Trying algorithm eng0{} for conv (f32[32,96,3,3]{3,2,1,0}, u8[0]{0}) custom-call(f32[8,96,512,512]{3,2,1,0}, f32[8,32,512,512]{3,2,1,0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardFilter", backend_config={"operation_queu

287/287 ━━━━━━━━━━━━━━━━━━━━ 0s 558ms/step - dice_coefficient: 0.1173 - hard_iou: 0.0931 - loss: 0.4702

2026-09-08 08:47:54.681965: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-08 08:47:54.898667: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-08 08:48:12.688138: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-08 08:48:12.889765: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-08 08:48:13.882295: E external/local_xla/xla/stream_


Epoch 1: val_hard_iou improved from None to 0.23922, saving model to /kaggle/working/localiser_run4_vgg16enc_patch/best.weights.h5

Epoch 1: finished saving model to /kaggle/working/localiser_run4_vgg16enc_patch/best.weights.h5
287/287 ━━━━━━━━━━━━━━━━━━━━ 286s 749ms/step - dice_coefficient: 0.2553 - hard_iou: 0.2023 - loss: 0.3984 - val_dice_coefficient: 0.3255 - val_hard_iou: 0.2392 - val_loss: 0.3529 - lesion_centred_frac: 0.5000 - fg_patch_frac: 0.9376 - lr_encoder: 2.0070e-05 - lr_decoder: 2.0070e-04
Epoch 2/100
287/287 ━━━━━━━━━━━━━━━━━━━━ 0s 558ms/step - dice_coefficient: 0.4165 - hard_iou: 0.3270 - loss: 0.3156
Epoch 2: val_hard_iou improved from 0.23922 to 0.28730, saving model to /kaggle/working/localiser_run4_vgg16enc_patch/best.weights.h5

Epoch 2: finished saving model to /kaggle/working/localiser_run4_vgg16enc_patch/best.weights.h5
287/287 ━━━━━━━━━━━━━━━━━━━━ 174s 605ms/step - dice_coefficient: 0.4182 - hard_iou: 0.3252 - loss: 0.3148 - val_dice_coefficient: 0.3934 - va

In [4]:
from pathlib import Path
from src import localise_eval
localise_eval.TENSORS = Path(LESION_STORE)
model = localise_eval._load_model(f"{OUT}/best.weights.h5", config.LOC_BASE_FILTERS,
                                  config.LOC_DEPTH, encoder=ENCODER or "none")
df = localise_eval.evaluate_split(model, "val")
rep = localise_eval.report(df, "val")
df.to_csv(f"{OUT}/localiser_boxes_val.csv", index=False)
json.dump([rep], open(f"{OUT}/localiser_metrics_val.json", "w"), indent=2)
print({k: round(rep[k], 4) for k in ("box_iou_inclusive_mean", "detection_rate",
                                     "box_iou_mean_given_detected", "detect_at_iou50", "dice_mean")})

[localise_eval] rebuilt vgg16-encoder U-Net (31,410,497 params) from /kaggle/working/localiser_run4_vgg16enc_patch/best.weights.h5


2026-09-08 13:32:10.347965: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-08 13:32:10.573424: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-08 13:32:42.639585: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-08 13:32:42.847708: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-08 13:32:44.525242: E external/local_xla/xla/stream_


=== val (n=243, scored=243, no GT=0) ===
  detection_rate                 0.9095
  fallback_rate                  0.0905
  mask_iou_mean                  0.4083
  dice_mean                      0.5102
  box_iou_inclusive_mean         0.4190
  box_iou_inclusive_median       0.5391
  box_iou_mean_given_detected    0.4607
  box_iou_mammogram_mean         0.4581
  detect_at_iou50                0.5391
  detect_at_iou30                0.5967
  centroid_hit_rate              0.6296
  box_iou_any_inclusive_mean     0.4580
  pct_rows_on_multilesion_images 0.1235
  box_iou_matched_k3_inclusive_mean 0.5065
  detect_at_iou50_matched_k3     0.6543
  detection_rate_matched_k3      0.7654
  n_rows_matched_box_differs_from_single 66.0000
  n_images                       225.0000
  dice_image_mean_inclusive      0.5391
  dice_image_mean_detected       0.5917
{'box_iou_inclusive_mean': 0.419, 'detection_rate': 0.9095, 'box_iou_mean_given_detected': 0.4607, 'detect_at_iou50': 0.5391, 'dice_mean': 0.510

In [5]:
print(sorted(os.listdir(OUT)))
shutil.make_archive(f"/kaggle/working/{RUN_NAME}", "zip", OUT)
print(f"-> /kaggle/working/{RUN_NAME}.zip")


['best.weights.h5', 'best_valloss.weights.h5', 'breast_boxes_train.npy', 'history.csv', 'localiser_boxes_val.csv', 'localiser_metrics_val.json', 'manifest.json']
-> /kaggle/working/localiser_run4_vgg16enc_patch.zip
